In [8]:
#!/usr/bin/env python3
import argparse
import csv
import numpy as np
import matplotlib.pyplot as plt

MANDATORY_COLUMNS = ["t", "po_x", "po_y", "po_z", "pd_x", "pd_y", "pd_z"]
VO_COLUMNS = ["vo_x", "vo_y", "vo_z"]
VD_COLUMNS = ["vd_x", "vd_y", "vd_z"]

DIST_THRESHOLDS = [
    ("Info", 2.0, "#4C78A8"),
    ("Warning", 1.2, "#F58518"),
    ("Emergency", 0.6, "#E45756"),
]

TTC_THRESHOLDS = [
    ("Info", 3.0, "#4C78A8"),
    ("Warning", 2.0, "#F58518"),
    ("Emergency", 1.0, "#E45756"),
]


def estimate_velocity(pos: np.ndarray, t: np.ndarray) -> np.ndarray:
    vel = np.zeros_like(pos)
    n = len(t)
    if n < 2:
        return vel
    dt = np.diff(t)
    dt = np.where(np.abs(dt) < 1e-9, 1e-9, dt)
    vel[1:] = (pos[1:] - pos[:-1]) / dt[:, None]
    vel[0] = vel[1]
    return vel


def load_csv_series(path: str):
    with open(path, "r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        if reader.fieldnames is None:
            raise ValueError("CSV header is missing.")

        col_map = {name.strip(): name for name in reader.fieldnames}

        missing = [c for c in MANDATORY_COLUMNS if c not in col_map]
        if missing:
            raise ValueError(f"Missing mandatory columns: {missing}")

        has_vo = all(c in col_map for c in VO_COLUMNS)
        has_vd = all(c in col_map for c in VD_COLUMNS)

        rows = list(reader)
        if not rows:
            raise ValueError("CSV has no data rows.")

        def col_values(col_name):
            raw_name = col_map[col_name]
            values = []
            for i, row in enumerate(rows, start=2):
                try:
                    values.append(float(row[raw_name]))
                except Exception as exc:
                    raise ValueError(f"Bad value at line {i}, column {col_name}") from exc
            return np.asarray(values, dtype=float)

        t = col_values("t")
        p_o = np.column_stack([col_values("po_x"), col_values("po_y"), col_values("po_z")])
        p_d = np.column_stack([col_values("pd_x"), col_values("pd_y"), col_values("pd_z")])

        order = np.argsort(t)
        t = t[order]
        p_o = p_o[order]
        p_d = p_d[order]

        if has_vo:
            v_o = np.column_stack([col_values("vo_x"), col_values("vo_y"), col_values("vo_z")])[order]
        else:
            v_o = estimate_velocity(p_o, t)

        if has_vd:
            v_d = np.column_stack([col_values("vd_x"), col_values("vd_y"), col_values("vd_z")])[order]
        else:
            v_d = estimate_velocity(p_d, t)

        return t, p_o, v_o, p_d, v_d


def generate_demo_series(total_time=12.0, dt=0.05):
    t = np.arange(0.0, total_time + 1e-12, dt)

    # monitored object
    p_o = np.column_stack([
        0.20 * t,
        0.15 * np.sin(0.25 * t),
        np.zeros_like(t),
    ])

    # dynamic object: approaching trajectory with slight lateral motion
    p_d = np.column_stack([
        7.8 - 0.72 * t + 0.06 * np.sin(0.9 * t),
        1.4 * np.cos(0.35 * t) - 0.05 * t,
        np.zeros_like(t),
    ])

    v_o = estimate_velocity(p_o, t)
    v_d = estimate_velocity(p_d, t)
    return t, p_o, v_o, p_d, v_d


def compute_metrics(t, p_o, v_o, p_d, v_d, r_o, r_d, v_min, T_p):
    eps = 1e-9

    r = p_d - p_o
    v_rel = v_d - v_o

    r_norm = np.linalg.norm(r, axis=1)
    d_cur = np.maximum(0.0, r_norm - r_o - r_d)

    unit_r = r / (r_norm[:, None] + eps)
    v_close = -np.einsum("ij,ij->i", v_rel, unit_r)

    ttc = np.full_like(d_cur, np.inf, dtype=float)
    overlap = d_cur <= eps
    closing = v_close > v_min
    ttc[overlap] = 0.0
    valid = (~overlap) & closing
    ttc[valid] = d_cur[valid] / v_close[valid]

    v_rel_norm2 = np.einsum("ij,ij->i", v_rel, v_rel)
    numer = -np.einsum("ij,ij->i", r, v_rel)
    tau_raw = np.divide(numer, v_rel_norm2, out=np.zeros_like(numer), where=v_rel_norm2 > eps)
    tau_star = np.clip(tau_raw, 0.0, T_p)

    closest_vec = r + tau_star[:, None] * v_rel
    d_pred = np.maximum(0.0, np.linalg.norm(closest_vec, axis=1) - r_o - r_d)

    return d_cur, d_pred, ttc


def plot_timeline(t, d_cur, d_pred, ttc, output_prefix, ttc_cap=6.0, save_pdf=True):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6.2), sharex=True, dpi=150)

    # top: clearance
    ax1.plot(t, d_cur, lw=2.0, color="#2F4B7C", label="d_cur (current clearance)")
    ax1.plot(t, d_pred, lw=2.0, color="#1B998B", label="d_pred (predicted clearance)")
    for name, thr, color in DIST_THRESHOLDS:
        ax1.axhline(thr, ls="--", lw=1.2, color=color, alpha=0.9, label=f"{name} threshold")
    ax1.set_ylabel("Distance (m)")
    ax1.set_ylim(bottom=0.0)
    ax1.grid(alpha=0.25)
    ax1.legend(ncol=2, fontsize=8, frameon=False)

    # bottom: TTC
    ttc_show = np.where(np.isfinite(ttc), ttc, ttc_cap)
    ax2.plot(t, ttc_show, lw=2.0, color="#6A4C93", label="TTC")
    inf_mask = ~np.isfinite(ttc)
    if np.any(inf_mask):
        ax2.scatter(
            t[inf_mask],
            np.full(np.sum(inf_mask), ttc_cap),
            s=10,
            marker="x",
            color="#777777",
            label="TTC = inf (clipped)",
        )
    for name, thr, color in TTC_THRESHOLDS:
        ax2.axhline(thr, ls="--", lw=1.2, color=color, alpha=0.9, label=f"{name} threshold")
    ax2.set_ylabel("TTC (s)")
    ax2.set_xlabel("Time (s)")
    ax2.set_ylim(0.0, ttc_cap)
    ax2.grid(alpha=0.25)
    ax2.legend(ncol=2, fontsize=8, frameon=False)

    plt.tight_layout()

    out_files = []
    png_path = f"{output_prefix}.png"
    fig.savefig(png_path, dpi=300, bbox_inches="tight")
    out_files.append(png_path)

    if save_pdf:
        pdf_path = f"{output_prefix}.pdf"
        fig.savefig(pdf_path, bbox_inches="tight")
        out_files.append(pdf_path)

    plt.close(fig)
    return out_files


def main():
    parser = argparse.ArgumentParser(
        description="Plot TTC and predicted-clearance timeline."
    )
    parser.add_argument("--input", type=str, default="", help="CSV input path.")
    parser.add_argument("--output", type=str, default="ttc_pred_clearance_timeline",
                        help="Output prefix.")
    parser.add_argument("--r_o", type=float, default=0.5, help="Monitored object radius (m).")
    parser.add_argument("--r_d", type=float, default=0.4, help="Dynamic object radius (m).")
    parser.add_argument("--v_min", type=float, default=0.05, help="Closing speed threshold (m/s).")
    parser.add_argument("--T_p", type=float, default=2.0, help="Prediction horizon (s).")
    parser.add_argument("--ttc_cap", type=float, default=6.0, help="TTC display cap (s).")
    parser.add_argument("--demo_time", type=float, default=12.0, help="Demo duration (s).")
    parser.add_argument("--demo_dt", type=float, default=0.05, help="Demo sampling interval (s).")
    parser.add_argument("--no_pdf", action="store_true", help="Disable PDF output.")
    args = parser.parse_args()

    if args.input:
        t, p_o, v_o, p_d, v_d = load_csv_series(args.input)
        source = "csv"
    else:
        t, p_o, v_o, p_d, v_d = generate_demo_series(args.demo_time, args.demo_dt)
        source = "demo"

    d_cur, d_pred, ttc = compute_metrics(
        t=t, p_o=p_o, v_o=v_o, p_d=p_d, v_d=v_d,
        r_o=args.r_o, r_d=args.r_d, v_min=args.v_min, T_p=args.T_p
    )

    out_files = plot_timeline(
        t=t, d_cur=d_cur, d_pred=d_pred, ttc=ttc,
        output_prefix=args.output, ttc_cap=args.ttc_cap,
        save_pdf=(not args.no_pdf)
    )

    finite_ttc = ttc[np.isfinite(ttc)]
    min_ttc_text = "inf" if finite_ttc.size == 0 else f"{np.min(finite_ttc):.3f}"

    print(f"source: {source}")
    print(f"frames: {len(t)}")
    print(f"min d_cur: {np.min(d_cur):.3f} m")
    print(f"min d_pred: {np.min(d_pred):.3f} m")
    print(f"min TTC: {min_ttc_text} s")
    print("saved files:")
    for p in out_files:
        print("  " + p)


if __name__ == "__main__":
    main()

usage: ipykernel_launcher.py [-h] [--input INPUT] [--output OUTPUT]
                             [--r_o R_O] [--r_d R_D] [--v_min V_MIN]
                             [--T_p T_P] [--ttc_cap TTC_CAP]
                             [--demo_time DEMO_TIME] [--demo_dt DEMO_DT]
                             [--no_pdf]
ipykernel_launcher.py: error: unrecognized arguments: --f=/Users/cyf/Library/Jupyter/runtime/kernel-v307358e8b0825f07359b111b0b3cb8e3d8d7a7593.json


SystemExit: 2

/Users/cyf/Library/Python/3.9/lib/python/site-packages/IPython/core/interactiveshell.py:3558: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [6]:
pip install matplotlib

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 7.8 MB 3.1 MB/s eta 0:00:01
     |████████████████████████████████| 249 kB 2.7 MB/s eta 0:00:01
     |████████████████████████████████| 2.9 MB 6.6 MB/s eta 0:00:01
     |████████████████████████████████| 122 kB 9.1 MB/s eta 0:00:01
     |████████████████████████████████| 64 kB 4.6 MB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


NameError: name 'Library' is not defined